# Loading and sanity checks
check that the data is what we think it is before building anything on it. Nothing is transformed in this notebook.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.config import DATA_PATH

pd.set_option('display.max_columns', None)

## Shape and first rows

In [2]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
assert df.shape == (100_000, 20), df.shape
df.head()

(100000, 20)


,Employee_ID,Department,Gender,Age,Job_Title,Hire_Date,Years_At_Company,Education_Level,Performance_Score,Monthly_Salary,Work_Hours_Per_Week,Projects_Handled,Overtime_Hours,Sick_Days,Remote_Work_Frequency,Team_Size,Training_Hours,Promotions,Employee_Satisfaction_Score,Resigned
0,1,IT,Male,55,Specialist,2022-01-19 08:03:05.556036,2,High School,5,6750.0,33,32,22,2,0,14,66,0,2.63,False
1,2,Finance,Male,29,Developer,2024-04-18 08:03:05.556036,0,High School,5,7500.0,34,34,13,14,100,12,61,2,1.72,False
2,3,Finance,Male,55,Specialist,2015-10-26 08:03:05.556036,8,High School,3,5850.0,37,27,6,3,50,10,1,0,3.17,False
3,4,Customer Support,Female,48,Analyst,2016-10-22 08:03:05.556036,7,Bachelor,2,4800.0,52,10,28,12,100,10,0,1,1.86,False
4,5,Engineering,Female,36,Analyst,2021-07-23 08:03:05.556036,3,Bachelor,2,4800.0,38,11,29,13,100,15,9,1,1.25,False


## Types

In [3]:
df.dtypes

Employee_ID                      int64
Department                      object
Gender                          object
Age                              int64
Job_Title                       object
Hire_Date                       object
Years_At_Company                 int64
Education_Level                 object
Performance_Score                int64
Monthly_Salary                 float64
Work_Hours_Per_Week              int64
Projects_Handled                 int64
Overtime_Hours                   int64
Sick_Days                        int64
Remote_Work_Frequency            int64
Team_Size                        int64
Training_Hours                   int64
Promotions                       int64
Employee_Satisfaction_Score    float64
Resigned                          bool
dtype: object

## Missing values and duplicates

In [4]:
print(df.isna().sum()[lambda s: s > 0])
print('missing total :', int(df.isna().sum().sum()))
print('duplicated rows :', int(df.duplicated().sum()))
if 'Employee_ID' in df.columns:
    print('Employee_ID unique :', df['Employee_ID'].is_unique)

Series([], dtype: int64)
missing total : 0
duplicated rows : 0
Employee_ID unique : True


## Numeric and categorical columns

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Employee_ID,100000.0,50000.500000,28867.657797,1.0,25000.75,50000.5,75000.25,100000.0
Age,100000.0,41.029410,11.244121,22.0,31.00,41.0,51.00,60.0
Years_At_Company,100000.0,4.476070,2.869336,0.0,2.00,4.0,7.00,10.0
Performance_Score,100000.0,2.995430,1.414726,1.0,2.00,3.0,4.00,5.0
Monthly_Salary,100000.0,6403.211000,1372.508717,3850.0,5250.00,6500.0,7500.00,9000.0
Work_Hours_Per_Week,100000.0,44.956950,8.942003,30.0,37.00,45.0,53.00,60.0
Projects_Handled,100000.0,24.431170,14.469584,0.0,12.00,24.0,37.00,49.0
Overtime_Hours,100000.0,14.514930,8.664026,0.0,7.00,15.0,22.00,29.0
Sick_Days,100000.0,7.008550,4.331591,0.0,3.00,7.0,11.00,14.0
Remote_Work_Frequency,100000.0,50.090500,35.351157,0.0,25.00,50.0,75.00,100.0


In [6]:
df.select_dtypes(include='object').nunique().sort_values()

Gender                3
Education_Level       4
Job_Title             7
Department            9
Hire_Date          3650
dtype: int64

## Target distribution

In [7]:
counts = df['Performance_Score'].value_counts().sort_index()
pd.DataFrame({'count': counts, 'share': (counts / len(df)).round(4)})

,count,share
Performance_Score,,
1,20120,0.2012
2,20013,0.2001
3,19999,0.2000
4,19940,0.1994
5,19928,0.1993


In [8]:
assert set(df['Performance_Score'].unique()) == {1, 2, 3, 4, 5}

## Findings

- **Shape**: (100000, 20), as expected.
- **Missing values / duplicates**: none. `Employee_ID` is unique (1 to 100000), so it is just an index and carries no information.
- **Types**: 5 text columns (`Department`, `Gender`, `Job_Title`, `Education_Level`, `Hire_Date`), `Resigned` is boolean, the rest is numeric. `Hire_Date` is stored as text (3650 distinct values) and has to be parsed as a date.
- **Target**: `Performance_Score` takes the values 1 to 5, with about 20,000 rows each (20120 / 20013 / 19999 / 19940 / 19928). The dataset is balanced, so any imbalance later is created on purpose. These counts give 402 and 4002 rows for classes 1 and 2 after keeping 2 % and 20 %.
- **Synthetic look**: the numeric columns look uniformly distributed (means close to the middle of each range, quartiles at 25 / 50 / 75 % of the range), e.g. `Age` 22-60, `Training_Hours` 0-99, `Promotions` 0-2. `Hire_Date` always has the same time of day. The data is most likely generated, which can explain why few features relate to the target.
- **To check in the EDA**: `Gender` has 3 categories, and `Monthly_Salary` seems to take few distinct values.